# Comparación de regresión logística, KNN y árbol de decisión para clasificar retinopatía diabética en imágenes de fondo de ojo

**Proyecto Integrador — IS-484 Inteligencia Artificial I** · Escuela Profesional de Ingeniería de Sistemas, UNSCH · 2026-II
**Autor:** Jhonatan Conde Taquiri · **Docente:** Dr. Hubner Janampa Patilla

Este notebook recorre las **17 etapas** del proyecto integrador. Usa el mismo código de `src/` y las tablas de `datos/` y `resultados/`.

**Cómo ejecutarlo:** abrir este archivo desde la carpeta `notebooks/` con el entorno `.venv` activado y usar *Kernel → Restart & Run All*.

**Dos reglas que respeta:**
- **No vuelve a abrir el conjunto de prueba para decidir nada.** La prueba se evaluó una sola vez en la Fase 6 (`src/fase6_paso2_prueba.py`, con candado); aquí solo se leen y verifican esos resultados guardados.
- **No muestra fotos del dataset**, porque las reglas de APTOS 2019 prohíben redistribuirlas.

In [ ]:
import os, sys, platform
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from IPython.display import Image, display

# Trabajar siempre desde la carpeta raiz del proyecto y usar los modulos de src/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ / "src"))
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

def tabla(ruta, n=None):
    # Muestra una tabla de resultados/ o datos/ si existe
    p = Path(ruta)
    if not p.exists():
        print(f"(no se encontro {ruta})")
        return
    df = pd.read_csv(p)
    display(df if n is None else df.head(n))

def figura(nombre, ancho=720):
    # Muestra una figura de figuras/ si existe
    p = Path("figuras") / nombre
    if p.exists():
        display(Image(filename=str(p), width=ancho))
    else:
        print(f"(no se encontro figuras/{nombre})")

print("Carpeta del proyecto:", RAIZ.name)
print(f"Python {platform.python_version()} | scikit-learn {sklearn.__version__} | pandas {pd.__version__} | numpy {np.__version__}")

## 1. Identificación y descripción del problema

La retinopatía diabética (RD) es una complicación de la diabetes que daña los vasos sanguíneos de la retina y puede llevar a la pérdida de la visión. Se diagnostica revisando fotografías de fondo de ojo, una tarea que requiere especialistas.

**Problema:** clasificar automáticamente una fotografía de fondo de ojo como **sin RD** o **con RD** a partir de características medibles en la imagen.

## 2. Objetivo de la solución de IA

Comparar tres clasificadores clásicos —**regresión logística, KNN y árbol de decisión**— en la clasificación binaria sin RD / con RD sobre APTOS 2019, usando **9 características explicables** construidas a partir de las imágenes, con un procedimiento **sin fuga de datos**, y evaluar si el **PCA** mejora los modelos.

## 3. Tipo de aprendizaje

**Aprendizaje supervisado, tarea de clasificación binaria** (Unidad II): cada foto tiene la etiqueta de un especialista.

- $y = 0$: sin RD (nivel 0)
- $y = 1$: con RD (niveles 1 a 4: leve, moderada, severa y proliferativa)

### Antecedentes (investigación formativa)

| Artículo | Datos | Método | Validación | Resultado |
|---|---|---|---|---|
| Butt et al. (2022) | APTOS, 3,662 imágenes | Características de GoogleNet + ResNet-18; SVM, RF, NB | No declara el protocolo | Binario 97.80 % (SVM) |
| Ali et al. (2020) | BVH (Pakistán), 2,500 imágenes | 245 características de textura reducidas a 13; SMO, LMT, SLg, MLP | Validación cruzada de 10 pliegues | 98.53 % a 99.73 % |
| Rahman et al. (2024) | APTOS 2019 en versión binaria | Características de ResNet-50, estandarización y PCA; SVM, KNN, HGB | 80/20 estratificado | SVM 96.9 %, KNN 95.6 % |
| Tăbăcaru et al. (2024) | Conteos de 5 niveles según el artículo (IDRiD) | Entropía de Shannon y difusa; 15 clasificadores | 70/30 con validación cruzada de 5 pliegues | Exactitud 0.929 y AUC 0.941 (sin RD frente a severa) |
| Uppamma y Bhattacharya (2023) | IDRiD + DIARETDB1 | Transformadas (DFT, DCT, Gabor, wavelets), selección MMFO y ensamble | 80:20 | 96.5 % (5 niveles) |

Las cifras de otros datasets o con características de redes profundas no son comparables directamente con las de este trabajo, que usa características hechas a mano y clasificadores clásicos.

## 4. Obtención y descripción del conjunto de datos

**APTOS 2019 Blindness Detection** (Kaggle): 3,662 fotografías de fondo de ojo con su nivel de RD (0 a 4). Las imágenes no se incluyen en el repositorio por la licencia.

Se detectaron **123 grupos de imágenes idénticas** (hash MD5), 46 de ellos repartidos entre las particiones originales (fuga de datos si no se tratan). Regla fijada antes de ver resultados: conservar una copia si la etiqueta binaria coincide y eliminar los 4 grupos con etiqueta contradictoria. Resultado: **3,530 imágenes únicas**.

In [ ]:
car = pd.read_csv("datos/caracteristicas.csv")
print("Imagenes unicas:", len(car))
display(pd.crosstab(car["diagnosis"], car["y"], margins=True, margins_name="Total")
          .rename_axis(index="nivel", columns="y (0 = sin RD, 1 = con RD)"))
p = Path("resultados/filas_eliminadas.csv")
if p.exists():
    print("Filas eliminadas por duplicados:", len(pd.read_csv(p)), "(3,662 - 3,530 = 132)")

## 5. Observaciones, características y variable objetivo

Cada **observación** es una fotografía. La **variable objetivo** es `y`. Las **9 características** se miden dentro de la retina recortada y llevada a 512 × 512, en el canal verde con CLAHE, sin el disco óptico:

| Característica | Qué mide | Cómo se obtiene |
|---|---|---|
| `area_rojas` | % de retina con lesiones rojas (microaneurismas, hemorragias) | Transformada *black-hat* y filtro de forma |
| `n_rojas` | Número de lesiones rojas | Conteo de regiones |
| `area_exudados` | % de retina con exudados (manchas brillantes) | Transformada *top-hat* fuera del disco óptico |
| `n_exudados` | Número de exudados | Conteo de regiones |
| `densidad_vasos` | Densidad de vasos sanguíneos | Filtro de Frangi y esqueleto de 1 píxel |
| `glcm_contraste` | Variación local de intensidad | Matriz de co-ocurrencia de niveles de gris (GLCM) |
| `glcm_homogeneidad` | Uniformidad de la textura | GLCM |
| `glcm_energia` | Regularidad de la textura | GLCM |
| `glcm_correlacion` | Dependencia lineal entre píxeles vecinos | GLCM |

No se usan el tamaño de la imagen, la proporción, el borde negro, el brillo ni la nitidez globales, porque delatan la cámara (ver etapa 6).

In [ ]:
from preprocesamiento import COLS
display(car[COLS].describe().T.round(3))

## 6. Análisis exploratorio

**Hallazgo central: el tamaño de la imagen casi determina la clase.** Cada cámara del dataset aporta casi siempre una sola clase, salvo el tamaño 2588 × 1958, que es mixto.

In [ ]:
eda = pd.read_csv("datos/eda_imagenes.csv")[["id_code", "tamano"]]
t = car[["id_code", "y"]].merge(eda, on="id_code")
resumen = (t.groupby("tamano")["y"].agg(fotos="size", pct_con_RD="mean")
             .assign(pct_con_RD=lambda d: (d["pct_con_RD"] * 100).round(1))
             .sort_values("fotos", ascending=False))
display(resumen)

In [ ]:
figura("fig_fase2_clase_por_tamano.png")
figura("fig_fase2_medidas_por_clase.png")
figura("fig_fase3_distribuciones.png")

Clasificar cada foto por la clase mayoritaria de su tamaño ya da **0.9034 de exactitud** en validación cruzada (Fase 2). Decisiones que se tomaron por esto:

1. No usar como características el tamaño, la proporción, el borde negro, el brillo ni la nitidez globales.
2. Reportar esta **línea base de tamaño** junto a los resultados.
3. Evaluar también el **subconjunto 2588 × 1958** (una sola cámara) como control.

## 7. Limpieza y preparación de datos (sin fuga)

- **División primero:** 80/20 estratificada, `random_state=42`, antes de mirar cualquier relación con la etiqueta.
- Todo lo que aprende de los datos (logaritmo, escalado, PCA) se ajusta **solo con entrenamiento**, dentro de un `Pipeline`.
- La prueba se usa **una sola vez**, al final.

In [ ]:
tr = pd.read_csv("datos/train.csv")
te = pd.read_csv("datos/test.csv")
print(f"Entrenamiento: {len(tr)} fotos ({tr['y'].mean() * 100:.2f} % con RD)")
print(f"Prueba:        {len(te)} fotos ({te['y'].mean() * 100:.2f} % con RD)")
assert set(tr["id_code"]).isdisjoint(te["id_code"]), "Hay fotos repetidas entre entrenamiento y prueba"
assert tr["id_code"].is_unique and te["id_code"].is_unique
print("OK: ninguna foto se repite entre entrenamiento y prueba")

Con el entrenamiento se revisó cada característica (AUC univariado), su comportamiento en una sola cámara y la correlación entre ellas. Se conservaron las 9.

In [ ]:
tabla("resultados/auc_univariado_train.csv")
figura("fig_fase4_caracteristicas_por_clase.png")
figura("fig_fase4_control_camara.png")
figura("fig_fase4_correlacion.png")

## 8. Codificación y escalamiento

- **Codificación:** no hace falta, todas las características son numéricas.
- **Logaritmo** $\ln(1 + x)$ en las 4 columnas de lesiones (muy asimétricas).
- **Estandarización z-score** $z = (x - \mu) / \sigma$, porque KNN, la regresión logística y el PCA dependen de la escala. $\mu$ y $\sigma$ se calculan solo con entrenamiento.

In [ ]:
from preprocesamiento import COLS_ORDEN_SALIDA, construir_preprocesamiento, construir_pipeline
pre = construir_preprocesamiento().fit(tr[COLS])          # aprende solo con entrenamiento
Z = pd.DataFrame(pre.transform(tr[COLS]), columns=COLS_ORDEN_SALIDA)
print("Entrenamiento transformado: media ~ 0 y desviacion ~ 1 en cada columna")
display(Z.describe().loc[["mean", "std"]].round(3))
tabla("resultados/efecto_log_train.csv")

## 9. Selección justificada de algoritmos

| Algoritmo | Cómo decide | Por qué se eligió |
|---|---|---|
| **Regresión logística** | $P(y = 1 \mid x) = \sigma(w^\top x + b)$, con $\sigma(z) = 1 / (1 + e^{-z})$ | Da probabilidades, permite mover el umbral (importante en medicina) y sus coeficientes son interpretables |
| **KNN** | Mayoría entre los $K$ vecinos más cercanos (distancia euclidiana) | No supone una frontera lineal; se ajusta solo con $K$ y los pesos |
| **Árbol de decisión** | Reglas *si-entonces* con umbrales que reducen la impureza de Gini $1 - \sum p_k^2$ | Reglas legibles, capta relaciones no lineales y no necesita escalar |
| **PCA** (complemento) | Proyecta sobre las direcciones de mayor varianza | Hay características muy correlacionadas (área y número de lesiones rojas $\rho = 0.971$) |

Los tres son modelos de la Unidad II del curso, comparables con el mismo preprocesamiento, las mismas particiones y la misma métrica.

## 10. Entrenamiento y ajuste de hiperparámetros

Validación cruzada estratificada de **5 pliegues** (`random_state=42`), solo con entrenamiento, con los mismos pliegues para todos los modelos. Métrica para elegir: **F1 medio**.

- Regresión logística: $C \in \{0.01, 0.1, 1, 10, 100\}$
- KNN: $K \in \{1, 3, 5, 7, 9, 11, 15, 21, 31, 51\}$, pesos `uniform` o `distance`
- Árbol: criterio `gini` o `entropy`, `max_depth` de 2 a 10, `min_samples_leaf` de 1 a 50
- Cada modelo con y sin PCA (90 % de varianza acumulada)

Configuraciones elegidas:

In [ ]:
tabla("resultados/configuraciones_elegidas.csv")

Verificación: se vuelve a correr la validación cruzada de las 3 configuraciones finales (solo entrenamiento). El F1 debe coincidir con el de la Fase 5.

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from validacion import crear_cv

X, y = tr[COLS], tr["y"].to_numpy()
FINALES = {
    "Regresion logistica": LogisticRegression(C=1, max_iter=1000),
    "KNN": KNeighborsClassifier(n_neighbors=11, weights="uniform"),
    "Arbol de decision": DecisionTreeClassifier(criterion="gini", max_depth=8, min_samples_leaf=5, random_state=42),
}
METRICAS = {"f1": "f1", "recall": "recall", "precision": "precision", "exactitud": "accuracy"}
filas = []
for nombre, modelo in FINALES.items():
    r = cross_validate(construir_pipeline(modelo), X, y, cv=crear_cv(), scoring=METRICAS)
    filas.append({"modelo": nombre, "F1": r["test_f1"].mean(), "F1_desv": r["test_f1"].std(),
                  "recall": r["test_recall"].mean(), "precision": r["test_precision"].mean(),
                  "exactitud": r["test_exactitud"].mean()})
cv = pd.DataFrame(filas).set_index("modelo")
cv["F1_fase5"] = pd.read_csv("resultados/configuraciones_elegidas.csv").set_index("modelo")["f1_media"]
display(cv.round(4))

In [ ]:
figura("fig_fase5_pca_varianza.png")
figura("fig_fase5_pca_2d.png")
figura("fig_fase5_knn_f1_vs_k.png")
figura("fig_fase5_arbol_f1_vs_profundidad.png")

**El PCA no mejoró ningún modelo** (igual en regresión logística, algo peor en KNN y claramente peor en el árbol), así que los modelos finales no lo usan.

**Umbral de la regresión logística:** regla fijada de antemano, el umbral más alto con recall ≥ 0.90 en validación cruzada → **t = 0.37**. Se reporta como fila adicional; la comparación principal usa 0.5 en los tres modelos.

In [ ]:
figura("fig_fase5_umbral_logreg.png")

## 11. Selección de métricas

Con la matriz de confusión (VP, FN, FP, VN):

- Exactitud $= \dfrac{VP + VN}{VP + VN + FP + FN}$
- Precisión $= \dfrac{VP}{VP + FP}$
- Recall (sensibilidad) $= \dfrac{VP}{VP + FN}$
- Especificidad $= \dfrac{VN}{VN + FP}$
- $F1 = \dfrac{2 \cdot \text{precisión} \cdot \text{recall}}{\text{precisión} + \text{recall}}$
- AUC: área bajo la curva ROC (complemento)

Se eligió el **F1** para comparar porque equilibra precisión y recall con una sola cifra. En medicina el **falso negativo** es el error más grave, por eso siempre se reporta el recall. Ejemplo numérico (VP = 40, FN = 10, FP = 5, VN = 45):

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
y_real = np.array([1] * 40 + [1] * 10 + [0] * 5 + [0] * 45)
y_pred = np.array([1] * 40 + [0] * 10 + [1] * 5 + [0] * 45)
print(f"Exactitud     = {accuracy_score(y_real, y_pred):.3f}")
print(f"Precision     = {precision_score(y_real, y_pred):.3f}")
print(f"Recall        = {recall_score(y_real, y_pred):.3f}")
print(f"Especificidad = {recall_score(y_real, y_pred, pos_label=0):.3f}")
print(f"F1            = {f1_score(y_real, y_pred):.3f}")

## 12. Evaluación y comparación (conjunto de prueba, una sola vez)

Los 3 modelos finales se entrenaron con todo el entrenamiento (2,824 fotos) y se evaluaron **una sola vez** en la prueba (706 fotos) en la Fase 6. Los intervalos de confianza al 95 % se calcularon con *bootstrap* (1,000 remuestreos).

In [ ]:
tabla("resultados/fase6_tabla_comparativa.csv")

Verificación de la tabla a partir de las probabilidades guardadas (no se reentrena ni se cambia nada):

In [ ]:
pred = pd.read_csv("resultados/fase6_predicciones_prueba.csv")
yt = pred["y"].to_numpy()
CASOS = [("Regresion logistica", 0.50), ("Regresion logistica", 0.37), ("KNN", 0.50), ("Arbol de decision", 0.50)]
filas = []
for m, t in CASOS:
    p = pred[f"p_{m}"].to_numpy()
    yp = (p >= t).astype(int)
    filas.append({"modelo": f"{m} (t = {t:.2f})", "exactitud": accuracy_score(yt, yp), "precision": precision_score(yt, yp),
                  "recall": recall_score(yt, yp), "especificidad": recall_score(yt, yp, pos_label=0),
                  "F1": f1_score(yt, yp), "AUC": roc_auc_score(yt, p)})
display(pd.DataFrame(filas).set_index("modelo").round(3))

Diferencias de F1 entre modelos (bootstrap pareado) y línea base de tamaño en la prueba:

In [ ]:
tabla("resultados/fase6_diferencias_bootstrap.csv")
tabla("resultados/fase6_linea_base_tamano.csv")

In [ ]:
figura("fig_fase6_matrices_confusion.png")
figura("fig_fase6_curvas_roc.png", ancho=560)
figura("fig_fase6_comparacion_camara.png")

**Lectura de los resultados:**

- KNN obtuvo la mayor exactitud (0.870) y F1 (0.869); solo su diferencia frente a la regresión logística es apenas clara (F1 +0.029, IC 95 % [0.004, 0.051]). KNN y el árbol son equivalentes.
- La regresión logística con umbral 0.37 reduce los falsos negativos de 61 a 36 (recall 0.896).
- La prueba coincide con la validación cruzada (diferencias de −0.020 a +0.001 en F1): no hay señales de fuga.
- La línea base de tamaño (0.897) supera a los modelos. En el subconjunto de una sola cámara (2588 × 1958, 103 fotos) los modelos siguen por encima de esa línea base (0.563), con exactitud de 0.78 a 0.80.

## 13. Interpretación de resultados

**¿Qué características usa el árbol?** Importancia por reducción de Gini, con su variación en los 5 pliegues:

In [ ]:
tabla("resultados/fase7_importancia_arbol.csv")
figura("fig_fase7_importancia_arbol.png")
figura("fig_fase7_arbol_ilustrativo.png", ancho=900)
figura("fig_fase5_logreg_coeficientes.png")

La **homogeneidad** (textura) y el **área de lesiones rojas** son las dos características más importantes en el árbol y las de mayor coeficiente en la regresión logística. El orden del resto no es estable entre pliegues. El número de lesiones rojas pesa poco en el árbol porque está muy correlacionado con el área ($\rho = 0.971$). El árbol de profundidad 3 es solo una ilustración, no el modelo final.

**¿Dónde se equivocan?** Acierto por nivel de gravedad en la prueba (nivel 0 = especificidad; niveles 1 a 4 = recall):

In [ ]:
niv = pd.read_csv("resultados/fase7_errores_por_nivel.csv")
display(niv.pivot(index="nivel", columns="modelo", values="tasa")
           [["LogReg t=0.50", "LogReg t=0.37", "KNN", "Arbol"]].round(3))
figura("fig_fase7_acierto_por_nivel.png")

In [ ]:
print("Errores por tamano de imagen (FP = sin RD marcada con RD; FN = con RD marcada sin RD):")
tabla("resultados/fase7_errores_por_tamano.csv")
comp = pd.read_csv("resultados/fase7_errores_compartidos.csv")
print("Fotos que fallaron los 3 modelos (umbral 0.5):")
display(pd.crosstab(comp["tipo"], comp["diagnosis"], margins=True, margins_name="Total")
          .rename_axis(index="tipo de error", columns="nivel"))

- Los niveles 2, 3 y 4 se clasifican bien (recall de 0.87 a 1.00). El **nivel 1** (el más leve) es el más difícil: recall de 0.52 a 0.73.
- Las fotos con RD que fallaron los 3 modelos son todas de nivel 1 o 2: casos con pocas lesiones visibles.
- Los errores se concentran en algunas cámaras: falsos positivos en 819 × 614 y falsos negativos en 1050 × 1050.

## 14. Identificación de limitaciones

- **Sesgo de cámara del dataset:** la línea base de tamaño (0.897) supera a los modelos, y en una sola cámara la exactitud baja a 0.78–0.80.
- **Rendimiento desigual** entre cámaras y menor en casos leves (nivel 1).
- **Características hechas a mano:** la textura puede depender de la nitidez de la foto; los exudados se comportan al revés de lo esperado.
- **Evaluación acotada:** un solo dataset y una sola partición, sin validación externa; la validación cruzada es levemente optimista porque las configuraciones se eligieron con ella.
- El umbral de recall 0.90 y el 90 % de varianza del PCA son decisiones del autor.

## 15. Análisis de aspectos éticos y riesgos

- **Riesgo clínico:** un falso negativo deja a un paciente con RD sin derivación; un falso positivo genera una derivación innecesaria. Este trabajo **no es una herramienta de diagnóstico** ni está validado clínicamente.
- **Sesgo y representatividad:** el dataset no tiene datos demográficos (edad, sexo), así que no se puede evaluar sesgo por grupos; el sesgo de cámara muestra que la forma de recolectar los datos influye en los resultados.
- **Licencia y privacidad:** las reglas de APTOS 2019 permiten el uso académico sin fines comerciales y prohíben redistribuir los datos; por eso el repositorio no incluye imágenes.

## 16. Conclusiones

1. Tres clasificadores clásicos con 9 características propias y explicables alcanzan en la prueba una exactitud de **0.85 a 0.87** y un AUC de **0.91 a 0.93**.
2. **KNN** obtuvo el mejor resultado, con una ventaja apenas clara sobre la regresión logística y equivalente al árbol de decisión.
3. Bajar el umbral de la regresión logística a **0.37** reduce los falsos negativos, lo que es preferible en un contexto médico.
4. El **PCA** no mejoró ningún modelo con estas 9 características.
5. El procedimiento evitó la fuga de datos: la prueba se usó una sola vez y sus resultados coinciden con la validación cruzada.
6. El dataset presenta un **sesgo de cámara** que debe controlarse al interpretar cualquier resultado sobre APTOS 2019; los casos leves (nivel 1) siguen siendo el principal reto.

## 17. Presentación y sustentación técnica

La sustentación es oral (8 de diciembre de 2026). Este notebook, el informe técnico y el repositorio de GitHub sirven de soporte.

## Referencias

Ali, A., Qadri, S., Mashwani, W. K., Kumam, W., Kumam, P., Naeem, S., Goktas, A., Jamal, F., Chesneau, C., Anam, S., & Sulaiman, M. (2020). Machine learning based automated segmentation and hybrid feature analysis for diabetic retinopathy classification using fundus image. *Entropy, 22*(5), Article 567. https://doi.org/10.3390/e22050567

Butt, M. M., Iskandar, D. N. F. A., Abdelhamid, S. E., Latif, G., & Alghazo, R. (2022). Diabetic retinopathy detection from fundus images of the eye using hybrid deep learning features. *Diagnostics, 12*(7), Article 1607. https://doi.org/10.3390/diagnostics12071607

Rahman, A., Youldash, M., Alshammari, G., Sebiany, A., Alzayat, J., Alsayed, M., Alqahtani, M., & Aljishi, N. (2024). Diabetic retinopathy detection: A hybrid intelligent approach. *Computers, Materials & Continua, 80*(3), 4561–4576. https://doi.org/10.32604/cmc.2024.055106

Tăbăcaru, G., Moldovanu, S., Răducan, E., & Barbu, M. (2024). A robust machine learning model for diabetic retinopathy classification. *Journal of Imaging, 10*(1), Article 8. https://doi.org/10.3390/jimaging10010008

Uppamma, P., & Bhattacharya, S. (2023). A multidomain bio-inspired feature extraction and selection model for diabetic retinopathy severity classification: An ensemble learning approach. *Scientific Reports, 13*, Article 18572. https://doi.org/10.1038/s41598-023-45886-7